In [ ]:
#@title 🚀 Ultimate ComfyUI VTON Studio (Final Production Build - Patched)
#@markdown เลือก Model ที่ต้องการติดตั้งลงใน ComfyUI
#@markdown ⚠️ หากต้องการเปลี่ยนโมเดล ให้ Factory Reset Runtime เครื่องก่อนเสมอ
TARGET_NODE = "IDM-VTON" #@param ["CatVTON", "IDM-VTON", "OOTDiffusion", "FitDiT"]

import os
import subprocess
import sys
import threading
import time
import socket

# 🛡️ SYSTEM HACK: ป้องกัน VRAM Fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def run_cmd(cmd, check=False):
    print(f"\n⚙️ Executing: {cmd}")
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        sys.stdout.write(line)
    process.wait()
    if check and process.returncode != 0:
        print(f"❌ Command failed: {cmd}")

print("🚀 [1/5] Preparing ComfyUI Core Environment...")
if not os.path.exists("/content/ComfyUI"):
    run_cmd("git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI")
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    run_cmd("git clone https://github.com/ltdrdata/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager")

os.chdir("/content/ComfyUI")
# 📦 ลง Dependencies หลักของ ComfyUI (แก้บั๊ก ModuleNotFoundError comfy_aimdo)
print("📦 [2/5] Installing ComfyUI Core Requirements & Global Vaccines...")
run_cmd("pip install -q -r requirements.txt")
run_cmd("pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")
run_cmd("pip install -q aria2")
# ☢️ GLOBAL VACCINE: กัน Numpy 2.0 Apocalypse พังระบบ (ชุบชีวิต onnxruntime)
run_cmd('pip install -q "numpy<2" onnxruntime kornia')

print(f"📦 [3/5] Installing Custom Node for: {TARGET_NODE}...")
os.chdir("/content/ComfyUI/custom_nodes")

if TARGET_NODE == "IDM-VTON":
    # โหลด Node ที่จำเป็นทั้งหมดล่วงหน้า (ไม่ต้องไปกด Manager ให้เสี่ยงพัง)
    required_nodes = [
        "https://github.com/TemryL/ComfyUI-IDM-VTON.git",
        "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
        "https://github.com/storyicon/comfyui_segment_anything.git"
    ]
    for url in required_nodes:
        repo_name = url.split("/")[-1].replace(".git", "")
        if not os.path.exists(repo_name):
            run_cmd(f"git clone {url}")

    print("🔨 [4/5] THE HAMMER FIX (IDM-VTON): Forcing stable dependencies...")
    # ล็อกเวอร์ชันเป๊ะๆ สำหรับ IDM-VTON และลงสมอง (segment-anything) ให้กล่อง SAM
    run_cmd("pip install -q diffusers==0.27.2 huggingface-hub==0.23.2 transformers==4.40.2 accelerate==0.30.0 xformers segment-anything")
    run_cmd("pip install -q -r ComfyUI-IDM-VTON/requirements.txt")
    run_cmd("pip install -q -r comfyui_controlnet_aux/requirements.txt")
    run_cmd("pip install -q -r comfyui_segment_anything/requirements.txt")

elif TARGET_NODE == "CatVTON":
    run_cmd("git clone https://github.com/chflame163/ComfyUI_CatVTON_Wrapper.git 2>/dev/null || true")
    os.chdir("ComfyUI_CatVTON_Wrapper")
    run_cmd("pip install -q -r requirements.txt")
    os.makedirs("../../models/checkpoints", exist_ok=True)
    if not os.path.exists("../../models/checkpoints/catvton.safetensors"):
        run_cmd("aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/zhy6769/CatVTON/resolve/main/catvton.safetensors -d ../../models/checkpoints -o catvton.safetensors")
    print("🔨 [4/5] THE HAMMER FIX (CatVTON): Forcing stable dependencies...")
    run_cmd("pip install -q huggingface-hub==0.23.2 diffusers==0.25.0 transformers==4.38.2")

elif TARGET_NODE == "OOTDiffusion":
    run_cmd("git clone https://github.com/AuroBit/ComfyUI-OOTDiffusion.git 2>/dev/null || true")
    os.chdir("ComfyUI-OOTDiffusion")
    run_cmd("pip install -q -r requirements.txt")
    os.makedirs("checkpoints", exist_ok=True)
    if not os.path.exists("checkpoints/human_parsing.bin"):
        run_cmd("aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/levihsu/OOTDiffusion/resolve/main/checkpoints/human_parsing.bin -d checkpoints -o human_parsing.bin")
    print("🔨 [4/5] THE HAMMER FIX (OOTDiffusion): Forcing stable dependencies...")
    run_cmd("pip install -q huggingface-hub==0.23.2 diffusers==0.25.0 transformers==4.38.2")

elif TARGET_NODE == "FitDiT":
    run_cmd("git clone https://github.com/BoyuanJiang/FitDiT-ComfyUI.git 2>/dev/null || true")
    os.chdir("FitDiT-ComfyUI")
    run_cmd("pip install -q -r requirements.txt")
    print("🔨 [4/5] THE HAMMER FIX (FitDiT): Forcing stable dependencies...")
    run_cmd("pip install -q huggingface-hub==0.23.2 diffusers==0.25.0 transformers==4.38.2")

print("🌐 [5/5] Launching Cloudflare Tunnel...")
os.chdir("/content")
run_cmd("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb")
run_cmd("dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1")

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if sock.connect_ex(('127.0.0.1', port)) == 0: break
        sock.close()

    p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("\n" + "🌟"*25)
            print(f"🚀 COMFYUI READY! คลิกลิงก์ด้านล่างเพื่อเข้าเว็บ 👉:")
            print(f"🔗 URL: {l[l.find('http'):].strip()}")
            print("🌟"*25 + "\n")

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

print("🔥 Starting ComfyUI Server...")
os.chdir("/content/ComfyUI")
# 🎯 รันด้วย Granular Precision (แก้บั๊ก ambiguous option --fp16)
!python main.py --dont-print-server --highvram --fp16-unet --fp16-vae